In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date, count, sum, when, current_timestamp
spark = SparkSession.builder.appName('_11').getOrCreate()

In [2]:
def runetl(last_successful_run_ts, customer_df, products_df, transactions_df, target_df_existing):
    # Filters
    # c.is_active = 1
    # t.is_deleted = 0 (false)
    # t.quantity > 0 (no negatives allowed)
    
    # Find Transaction count for each (id, date)
    # if transaction.quantity >= 10 then provide (10%*transaction.quantity*product.price) else 0 as discount
    # sales = quantity*price
    filtered_transaction_df = transactions_df \
                                .filter(col('transaction_ts')>last_successful_run_ts) \
                                .filter(col('quantity')>0) \
                                .filter(col('is_deleted')==0) \
                                .dropDuplicates()
    
    filtered_customer_df = customer_df.filter(col('is_active')==1)
    
    joined_df1 = filtered_transaction_df \
                    .join(filtered_customer_df, 'customer_id', how='inner') \
                    .join(products_df, 'product_id', how='inner')
    
    joined_df2 = joined_df1.select('customer_id', 'transaction_ts', 'transaction_id', 'quantity', 'unit_price')
    joined_df3 = joined_df2.withColumn('gross_sales', col('quantity')*col('unit_price')) \
                            .withColumn('discount_amount', when(col('quantity')>=10, col('gross_sales')*0.1).otherwise(0))
    
    joined_df4 = joined_df3.withColumn('sales_date', to_date(col('transaction_ts'))) \
                .groupBy('customer_id','sales_date') \
                .agg(
                    count('*').alias('txn_count'),
                    sum('gross_sales').alias('gross_sales'),
                    sum('discount_amount').alias('discount_amount')
                ) \
                .withColumn('etl_updated_at', current_timestamp())
                
    
    # merge or upsert ('incremental')
    existing_without_updates = target_df_existing.join(joined_df4, ['customer_id', 'sales_date'], how='left_anti')
    
    incremental_load = existing_without_updates.unionByName(joined_df4)
    
    incremental_load.show()

In [3]:


customer_df = spark.read \
                .option('header', True) \
                .option('inferSchema', True) \
                .csv('customers.csv')

etl_metadata = spark.read \
                .option('header', True) \
                .option('inferSchema', True) \
                .csv('etl_metadata.csv')


last_successful_run_ts = etl_metadata.collect()[0]['last_successful_run_ts']

products_df = spark.read \
                .option('header', True) \
                .option('inferSchema', True) \
                .option('mode', 'DropMalformed') \
                .csv('products.csv') \
                .dropna(how='any')

transactions_df = spark.read \
                    .option('header', True) \
                    .option('inferSchema', True) \
                    .csv('transactions.csv')

target_df_existing = spark.read \
                        .option('header', True) \
                        .option('inferSchema', True) \
                        .csv('target_summary.csv')

In [4]:
runetl(last_successful_run_ts, customer_df, products_df, transactions_df, target_df_existing)

+-----------+----------+---------+-----------+---------------+--------------------+
|customer_id|sales_date|txn_count|gross_sales|discount_amount|      etl_updated_at|
+-----------+----------+---------+-----------+---------------+--------------------+
|        101|2025-02-01|        3|        510|           48.0|2026-07-03 09:28:...|
|        101|2025-02-02|        1|         20|            0.0|2026-07-03 09:28:...|
+-----------+----------+---------+-----------+---------------+--------------------+

+-----------+----------+---------+-----------+---------------+--------------------+
|customer_id|sales_date|txn_count|gross_sales|discount_amount|      etl_updated_at|
+-----------+----------+---------+-----------+---------------+--------------------+
|        200|2025-02-01|        1|         30|            0.0| 2025-02-01 23:59:00|
|        101|2025-02-01|        3|        510|           48.0|2026-07-03 09:28:...|
|        101|2025-02-02|        1|         20|            0.0|2026-07-03 09